# Measure the Internet with Cloudflare Radar 🌍
**SINF 2026 · FEUP · Cloudflare workshop**

Every day, a huge share of the world's websites run through Cloudflare's network.
[Cloudflare Radar](https://radar.cloudflare.com) shares public trends from that traffic.

Today **you** will write Python to answer real questions about the Internet, starting with Portugal.

**Rules of the challenge**
- Work in pairs.
- Each challenge has a goal, a check, and 💡 hints. Open the hints only if you're stuck.
- Use the docs: [Radar API reference](https://developers.cloudflare.com/api/resources/radar/)

______________

### **Part 1: Setup and first test**

**Step 0: Get your Radar API token**

1. Create a free Cloudflare account at [dash.cloudflare.com/sign-up](https://dash.cloudflare.com/sign-up) and **verify your email**
2. Go to **My Profile** → **API Tokens** → **Create Token**
3. Under **Custom token**, click **Get started**
4. Fill in:
   - **Token name:** `sinf-workshop`
   - **Permissions:** `Account` → `Radar` → `Read`
5. Click **Continue to summary** → **Create Token**
6. **Copy the token.** It's shown only once.

**Step 1: Save your token safely**

Click the **key icon** on the left and add a secret:
- **Name:** `RADAR_TOKEN`
- **Value:** your Radar API token
- Turn on **Notebook access**

⚠️ Never paste your token directly in the code.

**Step 2: Load the libraries**

Run the cell below (**Shift + Enter**). It loads everything you need today.

In [ ]:
import requests
import pandas as pd
from google.colab import userdata

print("Ready!")

**Step 3: Your first request**

**Goal:** in the cell below, ask the Radar API for the **share of mobile vs desktop traffic in the world, over the last 7 days**.

**You got it if** you print something like: `{'desktop': '...', 'mobile': '...', 'other': '...'}`

**Docs that help:**
- [Make your first API request](https://developers.cloudflare.com/radar/get-started/first-request/): a step-by-step example
- [HTTP summary by dimension](https://developers.cloudflare.com/api/resources/radar/subresources/http/methods/summary_v2/): every parameter you can use (`dateRange`, `location`, `botClass`...)

<details><summary>Why you should use <code>requests</code> and not the Cloudflare SDK</summary>

Cloudflare has an official Python library (`pip install cloudflare`) and it's a great choice for real projects as it has autocomplete, types and retries.

Today we use plain `requests` on purpose:
- You see the **real HTTP request**: the URL, the header and the parameters.
- The parameters match the **API docs** one to one.
- `requests` already comes with Colab, so there's nothing to install.
- It works with **any** HTTP API, not just Cloudflare's.

</details>

<details><summary>💡 Hint 1: Where is the data?</summary>

The endpoint is:
`https://api.cloudflare.com/client/v4/radar/http/summary/DEVICE_TYPE`

Docs: [HTTP summary by dimension](https://developers.cloudflare.com/api/resources/radar/subresources/http/methods/summary_v2/)
</details>

<details><summary>💡 Hint 2: How do I read my secret and call the API?</summary>

- Read the token with `userdata.get("RADAR_TOKEN")`.
- Make the request with the `requests` library.
- The API expects a header: `Authorization: Bearer <your token>`
- The time window is a parameter: `dateRange`
</details>

<details><summary>💡 Hint 3: I get a big JSON, where's my answer?</summary>

Look inside `result`, then `summary_0`.
</details>

### **Build your own helper**

You'll call the API many times today. Let's not copy-paste the same code over and over.

**Goal:** write a function `radar(path, **params)` that:
- takes the part of the URL **after** `/radar/` (for example `"http/summary/DEVICE_TYPE"`)
- takes any filters as keyword arguments (for example `dateRange="7d"`)
- returns only the `result` part of the answer

**You got it if** this gives you the same answer as Part 1:
`radar("http/summary/DEVICE_TYPE", dateRange="7d")["summary_0"]`

<details><summary>💡 Hint 1</summary>

Start from your Part 1 code and turn it into a function. An f-string is useful to build the URL.
</details>

<details><summary>💡 Hint 2</summary>

`requests.get(...)` accepts `params=params`, so you don't need to build the `?a=b&c=d` part yourself.
</details>

<details><summary>💡 Hint 3: Bonus</summary>

Call `r.raise_for_status()` so errors show up clearly instead of failing silently later.
</details>

<details><summary>🆘 <b>Stuck? Use this helper</b></summary>

No problem! Copy this into the cell below, run it, and move on to Part 2. You can come back and understand it later.

```python
TOKEN = userdata.get("RADAR_TOKEN")

def radar(path, **params):
    """Call the Radar API and return the 'result' part."""
    r = requests.get(f"https://api.cloudflare.com/client/v4/radar/{path}",
                     params=params,
                     headers={"Authorization": f"Bearer {TOKEN}"})
    r.raise_for_status()
    return r.json()["result"]
```

</details>

## **Part 2: Ask real questions!!**

Now you have your own `radar()` tool. Time to ask real questions!

Each challenge gives you a **question**. You decide how to answer it with data.
There are no wrong answers, only interesting data.

____________

### **Challenge 1: Is Portugal more mobile than the world?**

**Question:** do people in Portugal use their phones more or less than the world average?

**Goal:** compare the share of **mobile vs desktop** traffic for **the world** and **Portugal** over the last 7 days, in a **table** and a **bar chart**.

**You got it if** you have a chart with two bars (World and Portugal) for each device type.

<details><summary>💡 Hint 1: How do I filter by country?</summary>

Look for a parameter called `location` in the docs. It uses 2-letter country codes (Portugal = `PT`).
</details>

<details><summary>💡 Hint 2: How do I make the table?</summary>

`pandas.DataFrame` accepts a dictionary of dictionaries: `{"World": ..., "Portugal": ...}`
</details>

<details><summary>💡 Hint 3: My chart is empty or broken!</summary>

Look closely at the values. Are they numbers or **text**?
`.astype(float)` might help.
</details>

🔍 **Explore:** try other countries (`ES`, `BR`, `IN`, `US`...). Which one is the most mobile?

**Look at your result.** Is mobile above or below 50%?

Most people browse on their phones... does your number feel right?

<details><summary>👉 <b>Click here once you've discussed it with your pair</b></summary>

<br>

### Challenge 1b: Who is really browsing? 🤖 vs 🧑

Not all Internet traffic comes from humans. A lot of it comes from **bots** like crawlers, scripts, monitoring tools...

**Question:** what happens to Challenge 1 if you count **humans only**?

**Before you code, guess:** will mobile go up or down?

**Goal:** repeat Challenge 1 with human traffic only.

**You got it if** your numbers change compared to Challenge 1.

<details><summary>💡 Hint 1</summary>

Search the docs for a parameter related to **bots**.

</details>

<details><summary>💡 Hint 2</summary>

The parameter is `botClass`. Which value means "human"?
[Bot classes](https://developers.cloudflare.com/radar/concepts/bot-classes/)

</details>

</details>

### **Challenge 2: The day Portugal went dark 🔌**

Something big happened to Portugal's Internet traffic in **April 2025**.

**Question:** can you find **what day** and **what time** it happened, just by looking at the data?

**Goal:** plot Portugal's **human** Internet traffic, **hour by hour**, for the **whole month of April 2025**.

**You got it if** you see a line chart with daily waves and **one day that looks very different**.

<details><summary>💡 Hint 1: Which data?</summary>

You need traffic **over time**, not a summary. Look in the docs for an HTTP **timeseries** endpoint.

</details>

<details><summary>💡 Hint 2: How do I choose the dates?</summary>

Instead of `dateRange`, use `dateStart` and `dateEnd`.
They use this format: `2025-04-01T00:00:00Z`

To get one point per hour, look for a parameter called `aggInterval`.

</details>

<details><summary>💡 Hint 3: Where are the numbers?</summary>

Inside `result` → `serie_0` you'll find two lists: `timestamps` and `values`.
`pd.to_datetime(...)` turns the timestamps into real dates for your chart.

</details>

<details><summary>💡 Hint 4: The times look wrong!</summary>

The API uses **UTC** time. Try `.tz_convert("Europe/Lisbon")` on your index. 🕐

</details>

🔍 **Explore:**
- At what **time** did the drop start?

🔍 **Explore:**
- How long did it take to come back to normal?

🔍 **Explore:**
- Try `ES`. Did Spain look the same?